# Flight Price Prediction: EDA and Regression Models
**Target:** `price`

**Models:** Simple Linear, Multiple Linear, Polynomial, Decision Tree, Random Forest

Place `train.csv` and `test.csv` in the same folder as this notebook (or change `DATA_DIR` below).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

RANDOM_STATE = 42
DATA_DIR = '.'
OUT = 'predictions.csv'

## 1. Load data

In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')

print('Train:', train.shape, '| Test:', test.shape)
train.head()

## 2. Exploratory Data Analysis

In [ ]:
train.info()
print('\nMissing values:\n', train.isna().sum())
print('\nDuplicate rows:', train.duplicated().sum())
print('Test missing values:', test.isna().sum().sum())

In [ ]:
train.describe().T

In [ ]:
train.describe(include='object').T

### 2.1 Target distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
sns.histplot(train['price'], bins=50, kde=True, ax=axes[0])
axes[0].set_title('Price')
sns.histplot(np.log1p(train['price']), bins=50, kde=True, ax=axes[1])
axes[1].set_title('log(1 + Price)')
sns.boxplot(x=train['price'], ax=axes[2])
axes[2].set_title('Price boxplot')
plt.tight_layout()
plt.show()

print('Skewness of price:', round(train['price'].skew(), 2))

### 2.2 Numeric features

In [ ]:
num_cols = ['duration', 'days_left']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for i, col in enumerate(num_cols):
    sns.histplot(train[col], bins=40, kde=True, ax=axes[0, i])
    axes[0, i].set_title(f'Distribution of {col}')
    sns.scatterplot(
        x=train[col], y=train['price'], hue=train['flight_class'],
        alpha=0.4, ax=axes[1, i]
    )
    axes[1, i].set_title(f'{col} vs price')

plt.tight_layout()
plt.show()

### 2.3 Categorical features vs price

In [ ]:
cat_cols = [
    'airline', 'source_city', 'departure_time', 'stops',
    'arrival_time', 'destination_city', 'flight_class'
]
fig, axes = plt.subplots(4, 2, figsize=(16, 18))

for ax, col in zip(axes.ravel(), cat_cols):
    order = train.groupby(col)['price'].median().sort_values().index
    sns.boxplot(data=train, x=col, y='price', order=order, ax=ax)
    ax.set_title(f'Price by {col}')
    ax.tick_params(axis='x', rotation=30)

axes.ravel()[-1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
print('Cardinality of "flight":', train['flight'].nunique(), '(too many levels, so it is dropped from modelling)')
train.groupby('flight_class')['price'].describe()

### 2.4 Correlation

In [ ]:
eda = train.drop(columns=['id', 'flight']).copy()
eda['stops_num'] = eda['stops'].map({'zero': 0, 'one': 1, 'two_or_more': 2})
eda['class_num'] = (eda['flight_class'] == 'Business').astype(int)

corr = eda[['duration', 'days_left', 'stops_num', 'class_num', 'price']].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation matrix')
plt.show()
corr['price'].drop('price').sort_values(key=abs, ascending=False)

### 2.5 Outliers (IQR rule)

In [ ]:
for c in ['price', 'duration']:
    q1, q3 = train[c].quantile([.25, .75]); iqr = q3 - q1
    n = ((train[c] < q1 - 1.5*iqr) | (train[c] > q3 + 1.5*iqr)).sum()
    print(f'{c}: {n} outliers ({n/len(train):.1%})')
print('\nOutliers are mostly Business-class fares and multi-stop long durations. They are genuine, so they are kept.')

**EDA takeaways** (see the plots above): `flight_class` is the dominant driver of price; stops, duration and airline also matter; `days_left` has a non-linear relationship with price (this motivates polynomial regression); price is right-skewed.

## 3. Preprocessing and train/validation split

In [ ]:
features = [
    'airline', 'source_city', 'departure_time', 'stops', 'arrival_time',
    'destination_city', 'flight_class', 'duration', 'days_left'
]
X, y = train[features], train['price']
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

cat_features = [col for col in features if not pd.api.types.is_numeric_dtype(X[col])]
num_features = ['duration', 'days_left']
print('Categorical:', cat_features)

def make_pre(scale=True, poly_degree=None):
    num_steps = []
    if poly_degree:
        num_steps.append(('poly', PolynomialFeatures(poly_degree, include_bias=False)))
    if scale:
        num_steps.append(('scale', StandardScaler()))

    num_transformer = Pipeline(num_steps) if num_steps else 'passthrough'
    return ColumnTransformer([
        ('num', num_transformer, num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ])

results = []

def evaluate(name, model):
    model.fit(X_train, y_train)
    predictions = model.predict(X_val)
    cv = cross_val_score(
        model, X_train, y_train,
        cv=KFold(5, shuffle=True, random_state=RANDOM_STATE),
        scoring='r2'
    ).mean()

    results.append({
        'Model': name,
        'R2 (val)': r2_score(y_val, predictions),
        'RMSE': np.sqrt(mean_squared_error(y_val, predictions)),
        'MAE': mean_absolute_error(y_val, predictions),
        'CV R2 (5-fold)': cv
    })
    return predictions

## 4. Simple Linear Regression (one feature)

In [ ]:
# Compare the numeric predictors and the binary flight-class feature.
for col in ['duration', 'days_left']:
    model = LinearRegression().fit(X_train[[col]], y_train)
    score = r2_score(y_val, model.predict(X_val[[col]]))
    print(f'{col:10s} R2 = {score:.3f}')

simple_X_train = pd.DataFrame({
    'is_business': (X_train['flight_class'] == 'Business').astype(int)
})
simple_X_val = pd.DataFrame({
    'is_business': (X_val['flight_class'] == 'Business').astype(int)
})

slr = LinearRegression().fit(simple_X_train, y_train)
p_slr = slr.predict(simple_X_val)
slr_r2 = r2_score(y_val, p_slr)
print(f'is_business R2 = {slr_r2:.3f}  (strongest single feature)')

results.append({
    'Model': 'Simple Linear (flight_class)',
    'R2 (val)': slr_r2,
    'RMSE': np.sqrt(mean_squared_error(y_val, p_slr)),
    'MAE': mean_absolute_error(y_val, p_slr),
    'CV R2 (5-fold)': cross_val_score(
        LinearRegression(), simple_X_train, y_train, cv=5, scoring='r2'
    ).mean()
})
print(f'Price = {slr.intercept_:.1f} + {slr.coef_[0]:.1f} x is_business')

## 5. Multiple Linear Regression (all features)

In [ ]:
mlr = Pipeline([('pre', make_pre()), ('model', LinearRegression())])
p_mlr = evaluate('Multiple Linear', mlr)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(y_val, p_mlr, alpha=0.3)
axes[0].plot([0, y.max()], [0, y.max()], 'r--')
axes[0].set(xlabel='Actual', ylabel='Predicted', title='MLR: actual vs predicted')

axes[1].scatter(p_mlr, y_val - p_mlr, alpha=0.3)
axes[1].axhline(0, color='r', ls='--')
axes[1].set(xlabel='Predicted', ylabel='Residual', title='MLR: residuals')

plt.tight_layout()
plt.show()
pd.DataFrame(results)

## 6. Polynomial Regression\nDegree of the numeric features is chosen by cross-validation (categoricals are one-hot encoded).

In [ ]:
cv = KFold(5, shuffle=True, random_state=RANDOM_STATE)
deg_scores = {
    degree: cross_val_score(
        Pipeline([
            ('pre', make_pre(poly_degree=degree)),
            ('model', LinearRegression())
        ]),
        X_train, y_train, cv=cv, scoring='r2'
    ).mean()
    for degree in range(1, 6)
}

best_deg = max(deg_scores, key=deg_scores.get)
print({degree: round(score, 4) for degree, score in deg_scores.items()},
      '-> best degree:', best_deg)

plt.plot(list(deg_scores), list(deg_scores.values()), 'o-')
plt.xlabel('Polynomial degree')
plt.ylabel('CV R2')
plt.title('Degree selection')
plt.show()

poly = Pipeline([
    ('pre', make_pre(poly_degree=best_deg)),
    ('model', LinearRegression())
])
p_poly = evaluate(f'Polynomial (deg {best_deg})', poly)

## 7. Decision Tree Regressor

In [ ]:
dt_grid = GridSearchCV(
    Pipeline([
        ('pre', make_pre(scale=False)),
        ('model', DecisionTreeRegressor(random_state=RANDOM_STATE))
    ]),
    {
        'model__max_depth': [6, 10, 14, None],
        'model__min_samples_leaf': [2, 5, 10]
    },
    cv=5, scoring='r2', n_jobs=-1
).fit(X_train, y_train)

print('Best params:', dt_grid.best_params_, '| CV R2:', round(dt_grid.best_score_, 4))
dt = dt_grid.best_estimator_
p_dt = evaluate('Decision Tree', dt)

## 8. Random Forest Regressor

In [ ]:
rf_grid = GridSearchCV(
    Pipeline([
        ('pre', make_pre(scale=False)),
        ('model', RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1))
    ]),
    {
        'model__n_estimators': [100],
        'model__max_depth': [12, 20],
        'model__min_samples_leaf': [3]
    },
    cv=3, scoring='r2', n_jobs=-1
).fit(X_train, y_train)

print('Best params:', rf_grid.best_params_, '| CV R2:', round(rf_grid.best_score_, 4))
rf = rf_grid.best_estimator_
p_rf = evaluate('Random Forest', rf)

feature_names = rf.named_steps['pre'].get_feature_names_out()
importance = pd.Series(
    rf.named_steps['model'].feature_importances_, index=feature_names
).sort_values(ascending=False).head(12)
importance.iloc[::-1].plot.barh(figsize=(8, 5), title='Random Forest: top 12 feature importances')
plt.show()

## 9. Model comparison

In [ ]:
res = pd.DataFrame(results).sort_values('R2 (val)', ascending=False).reset_index(drop=True)
display(res.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=res, y='Model', x='R2 (val)', ax=axes[0])
axes[0].set_title('Validation R2 (higher is better)')
sns.barplot(data=res, y='Model', x='RMSE', ax=axes[1])
axes[1].set_title('RMSE (lower is better)')
plt.tight_layout()
plt.show()

best_name = res.loc[0, 'Model']
print('Best model:', best_name)

## 10. Predict on `test.csv`
The best model is refit on **all** training data and used to predict test prices.

In [ ]:
models = {
    'Multiple Linear': mlr,
    f'Polynomial (deg {best_deg})': poly,
    'Decision Tree': dt,
    'Random Forest': rf
}
final = models.get(best_name, rf)
final.fit(X, y)

submission = pd.DataFrame({
    'id': test['id'],
    'price': final.predict(test[features]).round(2)
})
submission.to_csv(OUT, index=False)
submission.head()

## 11. Conclusions
- **EDA:** no missing values or duplicates; price is right-skewed and driven mainly by `flight_class`, then `stops`, `duration` and `airline`.
- **Simple linear regression** on one feature explains only part of the variance; **multiple linear regression** improves on it but is limited by linearity.
- **Polynomial regression** captures the non-linear effect of `duration`/`days_left` and typically improves on plain MLR.
- **Tree-based models** capture interactions (e.g. class x airline x stops) and usually win; the **random forest** is more stable than a single tree.
- See the table in section 9 for the final ranking, and `predictions.csv` for test-set predictions.